# Benchmark 分析

用于加载 `train.py` 的训练产物并做基础评估分析。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

ARTIFACT_DIR = Path('./artifacts')
META_PATH = ARTIFACT_DIR / 'run_metadata.json'
PRED_DIR = ARTIFACT_DIR / 'predictions'

In [ ]:
with META_PATH.open('r', encoding='utf-8') as f:
    metadata = json.load(f)

df_train = pd.read_csv(PRED_DIR / 'train_predictions.csv')
df_val = pd.read_csv(PRED_DIR / 'val_predictions.csv')
df_test = pd.read_csv(PRED_DIR / 'test_predictions.csv')

print('model_path:', metadata.get('model_path', ''))
print('rows:', {k: len(v) for k, v in {'train': df_train, 'val': df_val, 'test': df_test}.items()})
df_test.head(3)

In [ ]:
def calc_metrics(df):
    y_true = df['target'].to_numpy()
    y_prob = df['prob'].to_numpy()
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {
        'rows': len(df),
        'bad_rate': float(df['target'].mean()),
        'auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'ks': float(np.max(tpr - fpr)),
        'score_min': float(df['score'].min()),
        'score_max': float(df['score'].max()),
    }

report = pd.DataFrame({
    'train': calc_metrics(df_train),
    'val': calc_metrics(df_val),
    'test': calc_metrics(df_test),
}).T

report

In [ ]:
def decile_report(df, score_col='score', target_col='target', q=10):
    tmp = df[[score_col, target_col]].copy()
    tmp['decile'] = pd.qcut(tmp[score_col], q=q, duplicates='drop')
    out = tmp.groupby('decile', observed=False).agg(
        rows=(target_col, 'size'),
        bads=(target_col, 'sum'),
        bad_rate=(target_col, 'mean'),
        score_min=(score_col, 'min'),
        score_max=(score_col, 'max'),
    ).reset_index()
    return out

test_decile = decile_report(df_test, q=10)
test_decile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_test['score'], bins=30)
axes[0].set_title('Test Score Distribution')
axes[0].set_xlabel('score')
axes[0].set_ylabel('count')

axes[1].plot(test_decile.index + 1, test_decile['bad_rate'], marker='o')
axes[1].set_title('Test Decile Bad Rate')
axes[1].set_xlabel('decile index')
axes[1].set_ylabel('bad_rate')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()